In [1]:
#读取数据并按时间排序
import pandas as pd
import numpy as np
import os

X = pd.read_csv("raw/secom.data", sep=" ", header=None)
X.columns = [f"f{i}" for i in range(X.shape[1])]
y_df = pd.read_csv("raw/secom_labels.data", sep=" ", header=None,
                   names=["label", "timestamp"])
y_df["timestamp"] = pd.to_datetime(y_df["timestamp"], dayfirst=True)

df = pd.concat([y_df, X], axis=1).sort_values("timestamp").reset_index(drop=True)
df["fail"] = (df["label"] == 1).astype(int)   # 不良 = 1，合格 = 0
feature_cols = list(X.columns)
print(df.shape)

(1567, 593)


In [2]:
#按时间划分，前70%训练，后30%测试
split_idx = int(len(df) * 0.7)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

for name, part in [("训练集", train), ("测试集", test)]:
    print(f"{name}: {part['timestamp'].min():%Y-%m-%d} 至 {part['timestamp'].max():%Y-%m-%d}，"
          f"样本 {len(part)}，不良 {part['fail'].sum()}，不良率 {part['fail'].mean():.2%}")

X_train, X_test = train[feature_cols], test[feature_cols]
y_train, y_test = train["fail"], test["fail"]

训练集: 2008-07-19 至 2008-09-26，样本 1096，不良 78，不良率 7.12%
测试集: 2008-09-26 至 2008-10-17，样本 471，不良 26，不良率 5.52%


In [5]:
#清洗（规则只从训练集学习）
# ① 删除常数特征（按训练集判断）
n_unique = X_train.nunique(dropna=True)
const_cols = n_unique[n_unique <= 1].index

# ② 删除缺失率 > 50% 的特征（按训练集判断）
missing_ratio = X_train.isna().mean()
high_missing = missing_ratio[missing_ratio > 0.5].index

group_A = ["f72", "f73", "f345", "f346"]   # 缺失与时间混杂，按探索阶段的决定删除
drop_1 = list(set(const_cols) | set(high_missing) | set(group_A))
X_train_1 = X_train.drop(columns=drop_1)
X_test_1 = X_test.drop(columns=drop_1)

# ③ 用训练集的中位数填补缺失值（测试集也用训练集的中位数）
medians = X_train_1.median()
X_train_2 = X_train_1.fillna(medians)
X_test_2 = X_test_1.fillna(medians)

# ④ 删除高度相关的冗余特征（相关系数绝对值 > 0.95，保留其中一个）
corr = X_train_2.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr = [c for c in upper.columns if (upper[c] > 0.95).any()]
X_train_clean = X_train_2.drop(columns=high_corr)
X_test_clean = X_test_2.drop(columns=high_corr)

print("原始特征数:", len(feature_cols))
print("① 常数特征:", len(const_cols))
print("② 高缺失特征:", len(high_missing))
print("③ 填补后仍有缺失？ 训练集:", X_train_2.isna().any().any(), "测试集:", X_test_2.isna().any().any())
print("④ 高相关冗余特征:", len(high_corr))
print("最终特征数:", X_train_clean.shape[1])

原始特征数: 590
① 常数特征: 122
② 高缺失特征: 24
③ 填补后仍有缺失？ 训练集: False 测试集: False
④ 高相关冗余特征: 172
最终特征数: 268


In [6]:
#保存清洗后的数据
os.makedirs("processed", exist_ok=True)

def save(X_part, part, path):
    out = X_part.copy()
    out["grpA_missing"] = part["f72"].isna().astype(int).values   # 候选特征，基线模型不使用
    out["fail"] = part["fail"].values
    out["timestamp"] = part["timestamp"].values
    out.to_csv(path, index=False)

save(X_train_clean, train, "processed/train_clean.csv")
save(X_test_clean, test, "processed/test_clean.csv")
print("已保存到 processed/ 文件夹")

已保存到 processed/ 文件夹


In [8]:
print("组A是否还在最终特征里:", [c for c in group_A if c in X_train_clean.columns])
print("最终特征数:", X_train_clean.shape[1])

组A是否还在最终特征里: []
最终特征数: 268
